# Importing necessary libraries

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

Mounted at /content/drive


# Load and Inspect the Data
First, look at the shape of your data and check for missing values.

In [32]:
# Load the training data
train_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projects/Easy/Titanic Survival Prediction Challenge/train.csv')
train_df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,3,male,NaN,0,0,56.4958,S
1,0,2,male,NaN,0,0,0.0000,S
2,0,1,male,NaN,0,0,221.7792,S
3,1,3,female,18.0,0,1,9.3500,S
4,1,2,female,31.0,1,1,26.2500,S


In [33]:
# Load the testing data
test_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projects/Easy/Titanic Survival Prediction Challenge/test_inputs.csv')
test_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,24.0,2,0,24.1500,S
1,3,male,44.0,0,1,16.1000,S
2,3,male,22.0,0,0,7.2250,C
3,3,male,41.0,2,0,14.1083,S
4,3,female,NaN,1,0,15.5000,Q


In [34]:
# Checking data ana data types
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Sex       712 non-null    object 
 3   Age       575 non-null    float64
 4   SibSp     712 non-null    int64  
 5   Parch     712 non-null    int64  
 6   Fare      712 non-null    float64
 7   Embarked  710 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 44.6+ KB


In [35]:
# Cheking for null values
train_df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,137
SibSp,0
Parch,0
Fare,0
Embarked,2


# Data Cleaning & Preprocessing
Real-world data has missing values. Usually, Age, Embarked, and Fare have gaps that will crash your model if left unhandled.

In [36]:
# Fill missing age values with mean from training data
age_median = train_df['Age'].median()
train_df['Age'] = train_df['Age'].fillna(age_median)
test_df['Age'] = test_df['Age'].fillna(age_median)

# Fill missing Embarked values with most commong port (mode) from training data
embarked_mode = train_df['Embarked'].mode()[0]
train_df['Embarked'] = train_df['Embarked'].fillna(embarked_mode)
test_df['Embarked'] = test_df['Embarked'].fillna(embarked_mode)

In [42]:
# Cheking for null values
train_df.isnull().sum()

,0
Survived,0
Pclass,0
Age,0
SibSp,0
Parch,0
Fare,0
Family_Size,0
Sex_male,0
Embarked_Q,0
Embarked_S,0


# Feature Engineering & Encoding
Machine learning models only understand numbers. We need to convert text columns (Sex, Embarked) into numerical values and combine family columns to give the model better patterns.

In [37]:
# Create a new feature: Total family-size abroad
train_df['Family_Size'] = train_df['SibSp'] + train_df['Parch'] + 1 # The + 1 is added to include the passenger themselves in the total count.
test_df['Family_Size'] = test_df['SibSp'] + test_df['Parch'] + 1

In [38]:
# Convert categorical text columns into numerical numbers (One-hot encoding)
features_encoding = ['Sex', 'Embarked']
train_df = pd.get_dummies(train_df, columns=features_encoding, drop_first=True)
test_df = pd.get_dummies(test_df, columns=features_encoding, drop_first=True)

In [39]:
# Align columns to make sure train and test have the exact same columns
train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)

In [40]:
train_df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Family_Size,Sex_male,Embarked_Q,Embarked_S
0,1,3,28.5,0,0,56.4958,1,True,False,True
1,0,2,28.5,0,0,0.0000,1,True,False,True
2,0,1,28.5,0,0,221.7792,1,True,False,True
3,1,3,18.0,0,1,9.3500,2,False,False,True
4,1,2,31.0,1,1,26.2500,3,False,False,True


In [41]:
test_df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Family_Size,Sex_male,Embarked_Q,Embarked_S
0,0,3,24.0,2,0,24.1500,3,True,False,True
1,0,3,44.0,0,1,16.1000,2,True,False,True
2,0,3,22.0,0,0,7.2250,1,True,False,False
3,0,3,41.0,2,0,14.1083,3,True,False,True
4,0,3,28.5,1,0,15.5000,2,False,True,False


# Separate Features and Target
Extract the target variable (Survived) from the training features. Drop columns that contain unique text or identifiers if they exist (like Name or Ticket), as they cause overfitting.

In [43]:
# Define features to use (exclude the target variable itself)
X_train = train_df.drop(columns=['Survived'])
y_train = train_df['Survived']

# Ensure the test set does not contain the target placeholder column
X_test = test_df.drop(columns=['Survived'])

# Train a Robust Classifier
A Random Forest Classifier is perfect here because it handles non-linear relationships well and resists overfitting.

In [58]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Initialize the model with tuned hyperparamaters to prevent overfitting
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Check estimated performance using cross validation
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
print(f"Estimated Training F1-Score: {cv_scores.mean():.2f}")

model.fit(X_train, y_train)

Estimated Training F1-Score: 0.73


RandomForestClassifier(max_depth=5, random_state=42)

# Generate Final Predictions
The final requirement states you must store the output as a NumPy array inside a variable called predictions.

In [52]:
# Generate prediction for the test sample
predictions = model.predict(X_test)

# Verify the output format matches the expected format
print(type(predictions))

# Display the first 10 predictions
predictions[:10]

<class 'numpy.ndarray'>


array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0])